# The metadata graph

BDF's metadata layer is being rebuilt around a graph rather than a document. This notebook covers
the three pieces that are in place: canonical data filenames, term resolution against a pinned
BattINFO context, and field-declared JSON-LD projection. The entity models built on top of them —
`Dataset`, `BatterySpec`, `TestProtocol`, `Acquisition`, `FileDataset` — arrive in later changes,
and `docs/examples/metadata.ipynb` still shows the dataclass API they replace.

Three properties hold throughout, and each is demonstrated below rather than asserted:

1. **Nothing is guessed.** Every ontology IRI comes from a key looked up in a version-pinned
   context. An unknown key raises; it never becomes an invented IRI.
2. **Nothing is named twice.** A field declares its own RDF mapping, so there is no separate
   serialiser to keep in step with the model.
3. **Everything is addressable.** Every node gets an identifier derived from the document's own
   location, so no subject is a blank node and other documents can point at it.

Everything here runs offline. No cell reaches the network.

## Canonical data filenames

A BDF data file names itself: `inst__cell__{date}_{test}.bdf.csv`, three `__`-separated segments
whose third fuses the run date and the test name on its first underscore. Ambient conditions and
replicate qualifiers belong inside the test name rather than in a fourth segment, so the segment
count stays fixed however elaborate the test description gets.

`bdf.filename` parses that convention into parts and formats parts back into a filename.

In [ ]:
from bdf import filename

parts = filename.parse("Microsoft__manufacturer1-endt44198-2024-A0006__202603_5xC5-25degC.bdf.csv")

print("institution:", parts.institution)
print("cell:       ", parts.cell)
print("date:       ", parts.date)
print("test:       ", parts.test)
print("ext:        ", parts.ext)

Formatting is the inverse of parsing the canonical shape, so a filename survives the round trip
unchanged — useful when a pipeline rewrites one segment (a corrected cell id, say) and has to put
the name back together.

In [ ]:
from dataclasses import replace

print(filename.format(parts))
print(filename.parse(filename.format(parts)) == parts)

# Rewriting a single segment leaves the rest of the name intact.
renamed = replace(parts, cell="manufacturer1-endt44198-2024-A0007")
print(filename.format(renamed))

An older four-segment variant (`inst__cell__date__NNN`, a bare run number in place of a fused test
name) still appears in existing datastores, so `parse` accepts it on read. `format` only ever emits
the canonical three-segment form, which is how a directory converges on the convention rather than
carrying both shapes forever. A name matching neither shape is a hard error rather than a silent
mis-split.

In [ ]:
legacy = filename.parse("Inst__Cell-01__20260301__001.bdf.csv")
print(legacy.date, legacy.test)
print(filename.format(legacy))

try:
    filename.parse("just-one-segment.bdf.csv")
except ValueError as exc:
    print("rejected:", exc)

## Resolving terms, never inventing them

Ontology IRIs are opaque: EMMO's identifier for a battery cycler is
`https://w3id.org/emmo/domain/battery#battery_23e6170d_...`, which no amount of string
concatenation on the label "BatteryCycler" will produce. `bdf.vocabulary` therefore resolves
*keys* against a vendored copy of BattINFO's own published context, pinned to one release.

In [ ]:
from bdf import vocabulary

print("pinned release:", vocabulary.BATTINFO_VERSION)
print("ontology IRI:  ", vocabulary.BATTINFO_ONTOLOGY_IRI)
print()
for key in ("BatteryCycler", "UpperVoltageLimit", "hasNumberValue"):
    print(f"{key:<20}", vocabulary.resolve(key))

The pin is not decoration. The vendored context version equals the `owl:imports` target of the
BDF ontology snapshot shipped in the same wheel, and a test asserts that equality, so the two
cannot drift apart silently. Resolution reads the vendored file from the installed package, so it
works with no network access — which is also why every cell in this notebook runs offline.

Resolution fails closed. An unknown key raises `UnknownTermError` with near matches rather than
falling back to a constructed IRI, because a plausible-looking wrong IRI is far more expensive to
discover downstream than an error at write time.

In [ ]:
try:
    vocabulary.resolve("UpperVoltageLimits")  # note the trailing 's'
except vocabulary.UnknownTermError as exc:
    print(exc.args[0])

Failing closed has a visible cost at this pin: terms published after 0.18.6 do not resolve, and
that is the intended behaviour rather than a bug to route around. `has_term` reports it without
raising, so a caller can degrade to a plain literal instead of aborting.

In [ ]:
for key in ("BatteryCycler", "BatteryCellSpecification", "INR18650"):
    print(f"{key:<26}", vocabulary.has_term(key))

`is_reference_term` asks a different question: does this predicate point at a resource, or at a
literal? BattINFO already answers it — its context marks object properties with `"@type": "@id"` —
so the projection layer reads that rather than restating it in Python.

In [ ]:
for key in ("hasMeasurementUnit", "hasControlParameter", "BatteryCycler"):
    print(f"{key:<22}", vocabulary.is_reference_term(key))

## Fields declare their own projection

With terms resolvable, a model can carry its RDF mapping on the fields themselves:
`Annotated[type, Marker(...)]`. One generic walker in `bdf.metadata_projection` turns any such
model into a graph and back, so there is no per-model serialiser to keep in step, and a field
carrying no marker is simply not projected.

In [ ]:
from datetime import date
from typing import Annotated, ClassVar

from pydantic import Field

from bdf.metadata_projection import (
    BdfModel,
    NodeRef,
    SameAs,
    Scalar,
    StrList,
    Typed,
    from_compact_jsonld,
    to_compact_jsonld,
)


class Lab(BdfModel):
    """An organisation, referenced from the note below."""

    _rdf_type: ClassVar[str | None] = "schema:Organization"

    name: Annotated[str, Scalar("schema:name")]
    ror: Annotated[str | None, SameAs(prefix="https://ror.org/")] = None


class Note(BdfModel):
    """A minimal dataset note. Every projected fact is declared on its field."""

    _rdf_type: ClassVar[str | None] = "schema:Dataset"

    name: Annotated[str, Scalar("schema:name")]
    published: Annotated[date | None, Typed("schema:datePublished", datatype="xsd:date")] = None
    keywords: Annotated[list[str], StrList("schema:keywords")] = Field(default_factory=list)
    publisher: Annotated[str | Lab | None, NodeRef(role="publisher", term="schema:publisher")] = None
    internal_note: str | None = None  # no marker, so never projected

In [ ]:
note = Note(
    name="HPPC at 25 degC",
    published=date(2026, 7, 27),
    keywords=["hppc", "inr21700"],
    publisher=Lab(name="Battery Data Alliance", ror="052gg0110"),
    internal_note="never leaves Python",
)

# The document base comes from the document's own name, so every identifier is
# relative to the file that carries it and no subject is ever a blank node.
text = to_compact_jsonld(note, base="notes.jsonld")
print(text)

Three things to notice in that output. The publisher is its own node at
`notes.jsonld#record/publisher` rather than an anonymous blob, so another document can point at it.
The two keywords are two direct `schema:keywords` edges, not an ordered RDF collection — RDF
asserts no order over repeated properties, so neither does BDF. And `internal_note` is absent: it
carries no marker.

In [ ]:
# Reading back is the same walker in reverse.
restored = from_compact_jsonld(Note, text)

print(restored.name)
print(restored.publisher)
print(restored.internal_note)  # None: it was never in the document

### Graphs are compared as graphs

Two serialisations of the same metadata differ in triple order and formatting, so comparing their
text answers the wrong question. Compare the graphs instead — `rdflib.compare.isomorphic` does it,
and BDF's round-trip contract is stated in exactly those terms: RDF equivalence, never byte
identity.

In [ ]:
from rdflib import Graph, Literal, URIRef
from rdflib.compare import isomorphic

# N-Triples requires absolute IRIs, so this comparison uses an absolute document
# IRI rather than the relative one a sidecar carries.
subject = URIRef("https://example.org/notes.jsonld#record")
graph = note.to_graph(subject=subject)

# The same graph in two syntaxes: different bytes, same facts.
turtle = graph.serialize(format="turtle")
ntriples = graph.serialize(format="nt")

reparsed_turtle = Graph().parse(data=turtle, format="turtle")
reparsed_ntriples = Graph().parse(data=ntriples, format="nt")

print("same bytes: ", turtle == ntriples)
print("same graph: ", isomorphic(reparsed_turtle, reparsed_ntriples))

In [ ]:
# One changed literal is still caught, so the comparison is not vacuous.
edited = Graph()
edited += graph
edited.remove((subject, URIRef("https://schema.org/name"), Literal("HPPC at 25 degC")))
edited.add((subject, URIRef("https://schema.org/name"), Literal("HPPC at 45 degC")))

print("same graph: ", isomorphic(graph, edited))